# Phân loại melanoma trên ảnh dermoscopy

Notebook sạch phục vụ tái lập pipeline nghiên cứu: kiểm tra metadata, chia dữ liệu theo bệnh nhân, tiền xử lý ảnh, huấn luyện MobileNetV2 và đánh giá classification/calibration.

> Chỉ dùng cho mục đích nghiên cứu, không dùng để chẩn đoán y khoa.


## 1. Cấu hình

Đặt biến môi trường `SIIM_ISIC_DATA_DIR` tới thư mục chứa `train.csv` và `jpeg/train/`. Nếu chạy trên Kaggle, notebook dùng đường dẫn cuộc thi mặc định.


In [ ]:
import os
import random
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from tensorflow.keras import mixed_precision

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import expected_calibration_error
from src.preprocessing import preprocess_image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_DIR = Path(
    os.getenv(
        "SIIM_ISIC_DATA_DIR",
        "/kaggle/input/competitions/siim-isic-melanoma-classification",
    )
)
OUTPUT_DIR = Path(os.getenv("OUTPUT_DIR", PROJECT_ROOT / "outputs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = DATA_DIR / "train.csv"
IMAGE_DIR = DATA_DIR / "jpeg" / "train"
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

print("TensorFlow:", tf.__version__)
print("Data:", DATA_DIR)


## 2. Đọc và kiểm tra metadata


In [ ]:
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Không tìm thấy {CSV_PATH}. Hãy đặt SIIM_ISIC_DATA_DIR đúng vị trí."
    )

df = pd.read_csv(CSV_PATH)
required_columns = {"image_name", "patient_id", "target"}
missing = required_columns.difference(df.columns)
if missing:
    raise ValueError(f"Thiếu các cột bắt buộc: {sorted(missing)}")

df = df.dropna(subset=["image_name", "patient_id", "target"]).copy()
df["target"] = df["target"].astype(int)
df["image_path"] = df["image_name"].map(lambda name: str(IMAGE_DIR / f"{name}.jpg"))

print(f"Số ảnh: {len(df):,}")
print(f"Số bệnh nhân: {df['patient_id'].nunique():,}")
display(df["target"].value_counts().rename(index={0: "benign", 1: "malignant"}))


In [ ]:
ax = sns.countplot(data=df, x="target")
ax.set_xticklabels(["Benign", "Malignant"])
ax.set(title="Phân bố nhãn", xlabel="Lớp", ylabel="Số ảnh")
plt.show()


## 3. Chia train/validation theo bệnh nhân

Không để ảnh của cùng một bệnh nhân xuất hiện ở cả train và validation.


In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, val_idx = next(splitter.split(df, y=df["target"], groups=df["patient_id"]))
train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)

assert set(train_df["patient_id"]).isdisjoint(set(val_df["patient_id"]))
print("Train:", train_df.shape, train_df["target"].value_counts().to_dict())
print("Validation:", val_df.shape, val_df["target"].value_counts().to_dict())


## 4. Oversampling lớp malignant trên tập train


In [ ]:
benign_train = train_df[train_df["target"] == 0]
malignant_train = train_df[train_df["target"] == 1]

if malignant_train.empty:
    raise ValueError("Tập train không có mẫu malignant.")

malignant_oversampled = malignant_train.sample(
    n=len(benign_train), replace=True, random_state=SEED
)
balanced_train_df = (
    pd.concat([benign_train, malignant_oversampled], ignore_index=True)
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)
print(balanced_train_df["target"].value_counts())


## 5. Pipeline ảnh với `tf.data`


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def _preprocess_numpy(path_bytes):
    path = path_bytes.decode("utf-8")
    image_bgr = cv2.imread(path)
    if image_bgr is None:
        raise FileNotFoundError(path)
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    return preprocess_image(image_rgb, IMAGE_SIZE)


def load_and_preprocess(path, label):
    image = tf.numpy_function(_preprocess_numpy, [path], Tout=tf.float32)
    image.set_shape((*IMAGE_SIZE, 3))
    return image, tf.cast(label, tf.float32)


def make_dataset(frame, training=False):
    dataset = tf.data.Dataset.from_tensor_slices(
        (frame["image_path"].values, frame["target"].values)
    )
    if training:
        dataset = dataset.shuffle(min(len(frame), 10_000), seed=SEED)
    dataset = dataset.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
    return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_ds = make_dataset(balanced_train_df, training=True)
val_ds = make_dataset(val_df)


## 6. MobileNetV2: warm-up và fine-tuning


In [ ]:
mixed_precision.set_global_policy("mixed_float16")

augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.08),
        tf.keras.layers.RandomZoom(0.10),
    ],
    name="augmentation",
)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(*IMAGE_SIZE, 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3))
x = augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x * 255.0)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.30)(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid", dtype="float32")(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.AUC(name="roc_auc"),
        tf.keras.metrics.AUC(curve="PR", name="pr_auc"),
    ],
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_roc_auc", mode="max", patience=3, restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        OUTPUT_DIR / "best_mobilenetv2.keras",
        monitor="val_roc_auc",
        mode="max",
        save_best_only=True,
    ),
]

history_warmup = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=callbacks,
)


In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.AUC(name="roc_auc"),
        tf.keras.metrics.AUC(curve="PR", name="pr_auc"),
    ],
)

history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks,
)


## 7. Đánh giá trên tập validation


In [ ]:
y_true = val_df["target"].to_numpy()
y_prob = model.predict(val_ds, verbose=1).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_true, y_prob))
print("PR-AUC:", average_precision_score(y_true, y_prob))
print(classification_report(y_true, y_pred, digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
ConfusionMatrixDisplay(cm, display_labels=["Benign", "Malignant"]).plot(cmap="Blues")
plt.title("Confusion matrix — validation")
plt.show()


## 8. Temperature scaling

Temperature được tối ưu trên logits của tập validation để minh họa quy trình. Với đánh giá nghiêm ngặt hơn, nên dành một calibration set riêng hoặc dùng nested cross-validation.


In [ ]:
epsilon = 1e-7
clipped = np.clip(y_prob, epsilon, 1 - epsilon)
logits = np.log(clipped / (1 - clipped)).astype(np.float32)

temperature_raw = tf.Variable(0.0, dtype=tf.float32)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
y_tensor = tf.constant(y_true.astype(np.float32))
logits_tensor = tf.constant(logits)

for _ in range(500):
    with tf.GradientTape() as tape:
        temperature = tf.nn.softplus(temperature_raw) + 1e-6
        loss = tf.reduce_mean(
            tf.nn.sigmoid_cross_entropy_with_logits(
                labels=y_tensor, logits=logits_tensor / temperature
            )
        )
    gradient = tape.gradient(loss, [temperature_raw])
    optimizer.apply_gradients(zip(gradient, [temperature_raw]))

temperature = float(tf.nn.softplus(temperature_raw).numpy() + 1e-6)
calibrated_prob = tf.sigmoid(logits_tensor / temperature).numpy()

print("Temperature:", temperature)
print("Brier trước:", brier_score_loss(y_true, y_prob))
print("Brier sau:", brier_score_loss(y_true, calibrated_prob))
print("ECE trước:", expected_calibration_error(y_true, y_prob))
print("ECE sau:", expected_calibration_error(y_true, calibrated_prob))


## 9. Ghi chú về độ bất định

MC Dropout có thể được bổ sung bằng cách giữ dropout hoạt động trong nhiều lần suy luận và đo trung bình/phương sai của xác suất. Repo này chưa báo cáo chỉ số uncertainty vì lần chạy gốc không lưu đủ output để xác nhận kết quả.
